# 02 — Create train/test batches and method-specific transformed matrices

Subjects are split into repeated train/test batches so all samples from one subject remain together.

Feature filtering is learned from training samples only:
- relative abundance >= 0.01
- prevalence >= 0.10

**TEMPTED input:** add a pseudocount of `0.5` to filtered counts, then apply ordinary CLR.

**MEFISTO input:** apply rCLR without a pseudocount. Original zeros remain missing (`NaN`) and are passed to MEFISTO as missing values.

Filtered raw counts are also retained for abundance-based plots.


In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

NUMBER_OF_BATCHES = 10
TRAIN_FRACTION = 0.70
MINIMUM_PREVALENCE = 0.10
MINIMUM_RELATIVE_ABUNDANCE = 0.01
TEMPTED_PSEUDOCOUNT = 0.5
RANDOM_SEED = 7319

root = Path(".") if Path("data").exists() else Path("..")
dataset = sorted(
    path for path in (root / "data" / "processed_16s").iterdir()
    if path.is_dir()
    and (path / "counts.csv").exists()
    and (path / "metadata.csv").exists()
)[-1]

output = root / "data" / "splits" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

counts = pd.read_csv(dataset / "counts.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(
    dataset / "metadata.csv",
    dtype={"sample_id": str, "subject_id": str, "label": str},
)
counts = counts.loc[metadata["sample_id"]]
subjects = metadata.drop_duplicates("subject_id")[["subject_id", "label"]]

print("Input:", dataset)
print("Output:", output)


## Method-specific preprocessing

### TEMPTED: `+0.5` then CLR

The TEMPTED paper applies CLR after adding a pseudocount of `0.5` to every microbiome count. Shi et al. justify `0.5` using a Dirichlet-multinomial bias argument and report that `0.5` and `1` give very similar performance, whereas `0.1` performs somewhat worse.

**Reference:** Shi P, Martino C, Han R, et al. *TEMPTED: time-informed dimensionality reduction for longitudinal microbiome studies.* **Genome Biology** 25, 317 (2024). https://doi.org/10.1186/s13059-024-03453-x

### MEFISTO: rCLR with missing zeros

MEFISTO does not use the TEMPTED pseudocount in this workflow. rCLR is computed using only positive entries in each sample, and original zeros remain missing rather than being imputed before model fitting.


In [ ]:
def relative_abundance(x):
    return x.div(x.sum(axis=1), axis=0)


def clr_with_pseudocount(x, pseudocount=0.5):
    values = np.log(x.to_numpy(float) + pseudocount)
    values = values - values.mean(axis=1, keepdims=True)
    return pd.DataFrame(values, index=x.index, columns=x.columns)


def rclr(x):
    values = x.to_numpy(float)
    transformed = np.full(values.shape, np.nan)

    for row_number, row in enumerate(values):
        positive = row > 0
        logged = np.log(row[positive])
        transformed[row_number, positive] = logged - logged.mean()

    return pd.DataFrame(transformed, index=x.index, columns=x.columns)


In [ ]:
rows = []

for number in range(1, NUMBER_OF_BATCHES + 1):
    train_subjects, test_subjects = train_test_split(
        subjects,
        train_size=TRAIN_FRACTION,
        stratify=subjects["label"],
        random_state=RANDOM_SEED + number,
    )

    train_meta = metadata[metadata["subject_id"].isin(train_subjects["subject_id"])].copy()
    test_meta = metadata[metadata["subject_id"].isin(test_subjects["subject_id"])].copy()
    train_counts = counts.loc[train_meta["sample_id"]]
    test_counts = counts.loc[test_meta["sample_id"]]

    keep = (
        relative_abundance(train_counts) >= MINIMUM_RELATIVE_ABUNDANCE
    ).mean(axis=0) >= MINIMUM_PREVALENCE

    features = train_counts.columns[keep]
    train_counts = train_counts[features]
    test_counts = test_counts[features]

    train_tempted_clr = clr_with_pseudocount(train_counts, TEMPTED_PSEUDOCOUNT)
    test_tempted_clr = clr_with_pseudocount(test_counts, TEMPTED_PSEUDOCOUNT)
    train_mefisto_rclr = rclr(train_counts)
    test_mefisto_rclr = rclr(test_counts)

    folder = output / f"batch_{number:03d}"
    folder.mkdir()

    for name, table in {
        "train_counts": train_counts,
        "test_counts": test_counts,
        "train_tempted_clr": train_tempted_clr,
        "test_tempted_clr": test_tempted_clr,
        "train_mefisto_rclr": train_mefisto_rclr,
        "test_mefisto_rclr": test_mefisto_rclr,
    }.items():
        table.rename_axis("sample_id").reset_index().to_csv(
            folder / f"{name}.csv.gz",
            index=False,
        )

    train_meta.to_csv(folder / "train_metadata.csv.gz", index=False)
    test_meta.to_csv(folder / "test_metadata.csv.gz", index=False)

    rows.append([
        folder.name,
        train_meta["subject_id"].nunique(),
        test_meta["subject_id"].nunique(),
        len(features),
        int(train_mefisto_rclr.isna().sum().sum()),
        int(test_mefisto_rclr.isna().sum().sum()),
    ])

summary = pd.DataFrame(
    rows,
    columns=[
        "batch",
        "train_subjects",
        "test_subjects",
        "features",
        "missing_train_mefisto_rclr",
        "missing_test_mefisto_rclr",
    ],
)
summary.to_csv(output / "batch_summary.csv", index=False)

print("Saved:", output)
summary
